# Wallet Strategy Selection

Unified selection stage: runs **all wallet-selection methods** and saves each
as a named `WalletSelectionStrategy` artifact in the workspace.  The backtest
stage loads these artifacts directly — no strategy logic is re-built there.

## Methods included
| id suffix | description |
|-----------|-------------|
| `skill_sweep` → `quality_core` | top-N by best skill metric (grid-searched on train-a→train-b) |
| `cohort_early_entry` | wallets that enter markets early |
| `cohort_smooth_pnl` | wallets with high PnL relative to volatility |
| `volatility` | volatility-filtered wallets (from the profitable_wallet_analysis path) |

Each method produces **three trigger variants**:
- `score_threshold`  — composite signal score ≥ calibrated threshold (Kelly sizing)
- `all_open_buys`    — every open-buy event (fixed stake baseline)
- `copy_triggers`    — every trade (open/add/close/reduce), tight slippage


In [441]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Configuration

In [442]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [443]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
# Split train_a / train_b at the trade-count median of training rows so that
# both halves always contain roughly equal numbers of trades even when the
# data is unevenly distributed over calendar time.
_train_rows = _sample.loc[_sample["is_train"]].sort_values("dt")
TRAIN_A_END_DATE = _train_rows["dt"].quantile(0.5, interpolation="nearest").date()
del _sample, _train_rows
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-02-15  TRAIN_A_END_DATE=2025-12-15


## Compute / load wallet skill metrics

In [444]:
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,10964
train_b_wallets,10964
full_train_wallets,10964
test_wallets,10964


## Cohort selection sweep (skill path)

Grid-search over metrics × top-N using train-a → train-b persistence.

In [445]:
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
10,avg_copy_roi_capped,50,50,1813,0.07,0.12,1.52,103284.02,0.42
11,avg_copy_roi_capped,100,100,2695,0.07,0.10,1.12,143225.93,0.42
0,prob_edge_shrunk,50,50,4204,0.01,0.12,0.93,115130.45,0.47
22,roi_sharpe,200,200,3250,0.01,0.07,0.85,101101.91,0.70
12,avg_copy_roi_capped,200,200,5272,0.06,0.11,0.82,248959.70,0.48
1,prob_edge_shrunk,100,100,5418,0.03,0.11,0.78,201409.43,0.51
13,avg_copy_roi_capped,300,300,7046,0.06,0.07,0.76,384078.84,0.50
23,roi_sharpe,300,300,4467,0.01,0.07,0.70,117444.18,0.70
6,weighted_prob_edge_shrunk,100,100,2399,0.06,0.09,0.68,256835.63,0.47
21,roi_sharpe,100,100,2473,0.01,0.03,0.66,73063.49,0.78


In [446]:
# pick best metric / top-N
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'avg_copy_roi_capped', 'best_top_n': 50}

## Select wallets (skill path) + build cohorts

## Build wallet profiles and signal events

In [447]:
# from polymarket_analysis.signal.builder import (
#     build_wallet_profiles,
#     build_signal_events,
# )

# selected_wallet_profiles = build_wallet_profiles(
#     dataset, selected_wallets, period="full_train",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
# )
# selected_wallet_profiles.to_parquet(
#     WORKSPACE_DIR / "selected_wallet_profiles_v2.parquet", index=False
# )

# # Force-regenerate signal caches: the schema now includes all event types
# # (open_buy, add_buy, close_sell, reduce_sell) plus copy_price/copy_market_key/
# # copy_token_winner columns. Delete stale parquets so they are always rebuilt.
# CALIBRATION_SIGNALS_PATH.unlink(missing_ok=True)
# TEST_SIGNALS_PATH.unlink(missing_ok=True)

# _, train_b_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="train_b",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# train_b_signals.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

# _, test_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="test",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# test_signals.to_parquet(TEST_SIGNALS_PATH, index=False)

# print(f"train_b signals: {len(train_b_signals):,}  test signals: {len(test_signals):,}")
# print(f"event types: {train_b_signals['event_type'].value_counts().to_dict()}")

## Calibrate signal scoring on train-B

In [448]:
# from polymarket_analysis.signal.scorer import (
#     build_calibration_tables,
#     apply_signal_score,
#     select_signal_threshold,
# )

# # Calibration tables are built from open_buy events only — the scoring model
# # is designed for open-buy conviction features. Non-open-buy rows get neutral
# # defaults and will not be used to calibrate the scorer.
# train_b_open_buys = train_b_signals[train_b_signals["event_type"] == "open_buy"].copy()

# price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
# train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
# SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
# print(f"Global edge: {global_edge:.4f}")
# print(f"Selected signal threshold: {SIGNAL_THRESHOLD:.2f}")

# # score distribution
# score_deciles = (
#     train_b_scored.assign(
#         score_decile=lambda df: pd.qcut(df["signal_score"], q=10, labels=False, duplicates="drop")
#     )
#     .groupby("score_decile")
#     .agg(
#         count=("signal_score", "size"),
#         avg_signal_score=("signal_score", "mean"),
#         avg_copy_roi_capped=("copy_roi_capped", "mean"),
#     )
#     .reset_index()
# )
# score_deciles

## Build wallet cohorts (skill path)

In [449]:
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts

# wallet_cohorts = build_wallet_cohorts(
#     full_train_metrics, train_b_open_buys, selected_wallets,
#     top_n=BEST_TOP_N,
# )
# {name: len(df) for name, df in wallet_cohorts.items()}


## Volatility-based wallet selection (second method)

Loads the full training set, applies the volatility filter, and stores the
result as an additional `WalletSelectionStrategy`.  The volatility wallet set
is added to `wallet_cohorts` so the strategy factory handles it uniformly.

In [450]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

# Load full training trades for the volatility path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_train_vol = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)
df_train_vol = df_train_vol[df_train_vol["is_train"]].copy()
df_train_vol["dt"] = pd.to_datetime(df_train_vol["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_train_vol.columns and "quantity" not in df_train_vol.columns:
    df_train_vol = df_train_vol.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_train_vol["usdc_amount"]      = df_train_vol["usdc_amount"].astype(float)
df_train_vol["final_value_usdc"] = df_train_vol["final_value_usdc"].astype(float)
df_train_vol["quantity"]         = df_train_vol["quantity"].astype(float)

df_train_vol["pnl"] = np.where(
    df_train_vol["side"] == "BUY",
    df_train_vol["final_value_usdc"] - df_train_vol["usdc_amount"],
    df_train_vol["usdc_amount"] - df_train_vol["final_value_usdc"],
)
df_train_vol["notional"] = np.where(
    df_train_vol["side"] == "BUY",
    df_train_vol["usdc_amount"],
    df_train_vol["quantity"] * (1 - df_train_vol["price"].astype(float)),
)
# ignore buys with large price, so we don't skew roi
# df_train_vol = df_train_vol[df_train_vol['price'] <= 0.95]
df_slice = df_train_vol[df_train_vol["is_train"]].copy()

wallet_vol_train, _ = compute_wallet_metrics(df_slice)
print(len(wallet_vol_train))
# del df_train_vol, df_slice

24500


In [451]:
wallet_vol_train['copyable_pnl_factor'] = np.clip(wallet_vol_train['copyable_pnl'] / wallet_vol_train['total_pnl'], 0, 1.0)
wallet_vol_train['copyable_roi'] = wallet_vol_train['average_roi'] * wallet_vol_train['copyable_pnl_factor']
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)
# wallet_vol_train = wallet_vol_train[wallet_vol_train['copyable_roi'] > 0.05]
# wallet_vol_train = wallet_vol_train[wallet_vol_train['top_market_pnl_pct'] < 0.3]

In [585]:
wallet_cohorts = {}

In [453]:
print(len(wallet_vol_train))
wallet_vol_train.head()


24500


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
0,0x63cbecd497f177b9a08c70b59db12d81959c65c0,1.72,4.00,3.00,209.19,230.21,230.21,1.00,0.99,1.10,28.08,0.00,0.00,0.00,0.00,1.10,1.00,28.08
1,0x389b5e9115f3af775e3f13b4c84b7b0e56583589,NaN,1.00,1.00,7.00,214.20,180.31,1.00,1.00,30.60,30.60,0.00,0.00,0.00,0.00,30.60,0.84,25.76
2,0x6a2c98ab174162a149980817eccadf4a9a980696,NaN,1.00,1.00,11.50,286.17,286.17,1.00,1.00,24.88,24.88,0.00,0.00,0.00,0.00,24.88,1.00,24.88
3,0xae5b076e1a5d41047819cea89919569c7c5d41cb,NaN,1.00,1.00,10.00,202.77,202.77,1.00,1.00,20.28,20.28,0.00,0.00,0.00,0.00,20.28,1.00,20.28
4,0xe5d0bc35e56d374e858366c4b83f491ad719e6a9,NaN,1.00,1.00,33.30,1090.55,643.64,1.00,1.00,32.75,32.75,0.00,0.00,0.00,0.00,32.75,0.59,19.33


In [454]:
# filtered_wallets_vol = filter_wallets_by_volatility(
#     wallet_vol_train,
#     min_buckets=20,
#     max_top5_pnl_pct=.6,
#     max_top_market_pnl_pct=0.5,
# )

# filtered_wallets_vol = wallet_vol_train.copy()

# filtered_wallets_vol = (
#     filtered_wallets_vol[
#         (filtered_wallets_vol['average_roi'] >= 0.04)
#         & (filtered_wallets_vol['copyable_pnl'] / filtered_wallets_vol['total_pnl'] >= 0.5)
# ].sort_values("pnl_volatility", ascending=True).head(100)
# )

# print(f"Volatility-filtered wallets: {len(filtered_wallets_vol)}")

# # Build wallet_quality based on total_pnl rank (higher = better)
# filtered_wallets_vol = filtered_wallets_vol.copy().reset_index(drop=True)
# filtered_wallets_vol["wallet_quality"] = filtered_wallets_vol["total_pnl"].rank(
#     method="first", pct=True
# )

# Add as additional cohort (uses only train-b signals for trigger calibration)
# We intersect with wallets that have signals to ensure coverage
# vol_wallets_with_signals = set(train_b_signals["wallet"]).intersection(
#     set(filtered_wallets_vol["wallet"])
# )
# [
#     filtered_wallets_vol["wallet"].isin(vol_wallets_with_signals)
# ][["wallet", "wallet_quality"]].copy().reset_index(drop=True)

In [455]:
wallet_vol_train.sort_values("copyable_pnl", ascending=False).head(10)

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
18612,0xdb27bf2ac5d428a9c63dbc914611036855a6c56e,47.61,7617.00,605.00,37824144.94,4923915.29,1290124.14,0.37,0.19,-1.00,-0.05,1668867.50,0.34,420627.87,0.33,0.13,0.26,-0.01
9235,0x743510ee9f21e24071c4e28edab4653df44ea620,251.02,1075.00,351.00,16726839.73,1136815.99,618751.25,1.53,0.81,0.69,0.24,1419183.12,1.25,513703.72,0.83,0.07,0.54,0.13
16989,0x9708fd9556116218fd29c778e9d81d255b61925b,92.21,2361.00,177.00,9446435.29,766847.15,608337.64,1.97,0.57,-1.00,0.01,686574.54,0.90,69019.29,0.11,0.08,0.79,0.00
13487,0xd38b71f3e8ed1af71983e5c309eac3dfa9b35029,51.08,814.00,437.00,10349270.48,2425353.99,600621.86,0.47,0.13,-1.00,0.18,514012.40,0.21,147570.58,0.25,0.23,0.25,0.04
12543,0x876426b52898c295848f56760dd24b55eda2604a,76.41,657.00,76.00,3927546.05,1554147.98,588511.05,1.05,0.44,-1.00,0.16,710281.07,0.46,130242.29,0.22,0.40,0.38,0.06
15187,0x14964aefa2cd7caff7878b3820a690a03c5aa429,54.99,8542.00,723.00,36380618.93,2649494.73,571057.19,0.45,0.32,0.47,0.11,1908470.23,0.72,264350.50,0.46,0.07,0.22,0.02
9887,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,24.96,755.00,71.00,4504209.75,1453742.72,550311.92,0.42,0.30,0.27,0.30,93020.86,0.06,90236.40,0.16,0.32,0.38,0.11
16525,0x16b29c50f2439faf627209b2ac0c7bbddaa8a881,50.37,29632.00,5503.00,69198139.42,3408554.38,458101.14,0.68,0.33,0.35,0.07,2084680.08,0.61,537082.37,1.17,0.05,0.13,0.01
10738,0x4bd74aef0ee5f1ec0718890f55c15f047e28373e,86.06,545.00,94.00,4340303.54,654851.06,436722.71,1.74,0.65,0.18,0.14,712417.48,1.09,98435.77,0.23,0.15,0.67,0.09
20733,0x42592084120b0d5287059919d2a96b3b7acb936f,290.24,1225.00,257.00,20322041.35,409492.65,401473.19,4.88,1.58,-0.99,-0.06,1520219.83,3.71,383171.89,0.95,0.02,0.98,-0.06


In [595]:
# wallet_cohorts['multi_filter'] = wallet_vol_train[
#     (wallet_vol_train['copyable_roi'] >= 0.04)
#     & (wallet_vol_train['num_buckets'] >= 20)
#     &(wallet_vol_train['copyable_pnl'] <= wallet_vol_train['total_pnl'])
#     &(wallet_vol_train['copyable_pnl_factor'] >=0.4)
#     # & (wallet_vol_train['median_roi'] >= 0)
#     & (wallet_vol_train['max_drawdown_to_pnl'] <= 0.3)
#     & (wallet_vol_train['max_copyable_drawdown_to_copyable_pnl'] <= 0.4)
#     # & (wallet_vol_train['num_markets'] >= 100)
#     & (wallet_vol_train['top_market_pnl_pct'] < 0.4)
#     & (wallet_vol_train['top5_pnl_pct'] < 0.5)
#     # & (wallet_vol_train['total_pnl'] < 10_000)
# ].sort_values("copyable_roi", ascending=False).copy().reset_index(drop=True)

wallet_cohorts['multi_filter'] = wallet_vol_train[
    (wallet_vol_train['copyable_roi'] >= 0.05)
    & (wallet_vol_train['copyable_pnl'] <= wallet_vol_train['total_pnl'])
    & (wallet_vol_train['max_drawdown_to_pnl'] <= 0.1)
    # & (wallet_vol_train['max_drawdown_to_pnl'] >= 0.1)
    & (wallet_vol_train['max_copyable_drawdown_to_copyable_pnl'] <= 0.4)
    & (wallet_vol_train['num_buckets'] >= 50)
    & (wallet_vol_train['top_market_pnl_pct'] < 0.3)
    & (wallet_vol_train['top5_pnl_pct'] < 0.5)
    & (wallet_vol_train['wallet'] != '0x17db3fcd93ba12d38382a0cade24b200185c5f6d')
    # & (wallet_vol_train['total_pnl'] > 10_000)
].sort_values("copyable_roi", ascending=False).copy().reset_index(drop=True)

len(wallet_cohorts['multi_filter'])

12

In [596]:
wallet_cohorts['multi_filter'].sort_values("copyable_pnl", ascending=False).head(20)

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
5,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,1.51,2084.00,568.00,163263.85,32073.32,8392.48,0.24,0.07,0.05,0.52,2085.63,0.07,1725.99,0.21,0.20,0.26,0.14
9,0x27b820e5203aa114acc2712e0e1d0ad758abb68c,2.20,789.00,251.00,230217.25,27609.69,7233.23,0.45,0.13,0.01,0.27,1045.61,0.04,1074.73,0.15,0.12,0.26,0.07
4,0x0cb10c40b0776e9ee8cef970af85724654dda76c,1.36,4029.00,1375.00,210082.01,32542.53,5767.43,0.18,0.09,0.01,1.07,1030.77,0.03,695.58,0.12,0.15,0.18,0.19
10,0x2853240a0f4e9e11a949a5cfa6e0fe953a293482,0.90,3884.00,1132.00,230662.67,29761.59,4503.21,0.20,0.11,0.02,0.44,744.59,0.03,559.70,0.12,0.13,0.15,0.07
8,0xc757850237e266d3b08d16b8455777a88590ccfc,1.62,166.00,44.00,1030198.18,5760.79,2787.96,0.32,0.27,0.01,0.17,496.00,0.09,496.00,0.18,0.01,0.48,0.08
1,0x672225e5e1aba1c970ac613efd1505f4b7a10762,1.55,770.00,240.00,175151.46,3742.51,1126.36,0.44,0.16,0.00,1.17,9.00,0.00,6.38,0.01,0.02,0.30,0.35
11,0x6b61b495ee20b5a161b7f75649e3b8888b217a33,3.52,149.00,25.00,11271.70,6048.95,783.70,0.46,0.27,0.44,0.47,190.05,0.03,33.62,0.04,0.54,0.13,0.06
6,0x5b7065359a6b6fe34d061b2e5a9bd06f2b5c5e72,2.02,59.00,57.00,5739.52,1973.45,710.06,0.29,0.06,0.79,0.35,196.00,0.10,108.00,0.15,0.34,0.36,0.13
2,0x520b4a0452005517964ee864985f2026d0dc100a,0.42,393.00,147.00,8354.05,1586.74,602.91,0.43,0.22,0.04,0.61,135.89,0.09,107.47,0.18,0.19,0.38,0.23
7,0x20a89c60befe1e2d866042deed93ab72be6fd960,0.51,968.00,334.00,8900.03,1925.32,594.18,0.26,0.07,0.22,0.31,97.65,0.05,89.93,0.15,0.22,0.31,0.10


In [559]:
df_slice.head()

,wallet,condition_id,dt,side,outcome,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count,pnl,notional
0,0x00046b741e19e9296863121475df84cc656b3746,0x0722102b758fff894d63c9d44ed47f24c99ca0486d746c82a5d06659c2e7d116,2026-01-09 08:26:42+00:00,BUY,Clippers,24.00,24.00,0.59,14.16,24.00,9.84,9.84,True,1.00,0x1c87b8cc6143ae2952e55266cefb1e7956a802b89bb0ff4aa2bf0308dc4e5e05,1,True,24.00,1478.88,15.00,9.84,14.16
7,0x000b88e5ff8880d41f87070a7bd8bab414220872,0x03d98588a81fdccc909ddf3734cbe3062e08f028c819b92117b63156c48df604,2025-05-07 03:27:36+00:00,BUY,Nationals,176.52,176.52,0.00,0.18,0.00,-0.18,0.00,False,0.00,0xf6c50ed35d141d582b919b1abed7a4211a910dbb3e97bd72c7b650a1c29346cf,1,True,0.00,0.00,0.00,-0.18,0.18
8,0x000b88e5ff8880d41f87070a7bd8bab414220872,0x052d57cb01437fefba9bef4c0fd06c4cd97acd628d193697bbd11f6c8bd62d68,2025-05-17 11:15:37+00:00,BUY,Yes,10.00,10.00,0.35,3.50,0.00,-3.50,-3.50,False,0.00,0x4c6d6082fbdb96b13cbac57be0a007f64c97946f595574b74eed1130ce0d8900,1,True,10.00,6.60,1.00,-3.50,3.50
9,0x000b88e5ff8880d41f87070a7bd8bab414220872,0x052d57cb01437fefba9bef4c0fd06c4cd97acd628d193697bbd11f6c8bd62d68,2025-05-17 11:20:03+00:00,SELL,Yes,0.00,10.00,0.34,3.40,0.00,3.40,0.00,False,0.00,0x8a504e00c218e3142786adc4121ce074dc7d40b796196fd4c0b277d7a4f06cea,1,True,-0.00,0.00,0.00,3.40,6.60
10,0x000b88e5ff8880d41f87070a7bd8bab414220872,0x027299cfb5a812c75ee6c0a037e2a6d135b1df3a97a0f149e088c3b45f84df97,2025-05-25 21:16:44+00:00,BUY,Mariners,20.00,20.00,0.00,0.02,0.00,-0.02,0.00,False,0.00,0x538515f3ffcfcbb7b0ff9949f2aeb8369e93793a1caef9b37a6c1681160033db,1,True,0.00,0.00,0.00,-0.02,0.02


In [598]:
wallets = set(wallet_cohorts['multi_filter']['wallet'])

df = df_train_vol
print(len(df))

df[df['wallet'].isin(wallets)].groupby('wallet').agg(
    pnl=('pnl', 'sum'),
    notional=('notional', 'sum'),
    buy_pnl=('pnl', lambda x: x[df.loc[x.index, 'side'] == 'BUY'].sum()),
    sell_pnl=('pnl', lambda x: x[df.loc[x.index, 'side'] == 'SELL'].sum()),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    buy_notional=('notional', lambda x: x[df.loc[x.index, 'side'] == 'BUY'].sum()),
    sell_notional=('notional', lambda x: x[df.loc[x.index, 'side'] == 'SELL'].sum()),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values(by="wallets", key=np.abs, ascending=False).head(50)

7111315


,pnl,notional,buy_pnl,sell_pnl,sell_count,buy_count,buy_notional,sell_notional,wallets
wallet,,,,,,,,,
0x0cb10c40b0776e9ee8cef970af85724654dda76c,32542.53,210082.01,35345.91,-2803.38,1425,3573,115457.70,94624.32,1
0x20a89c60befe1e2d866042deed93ab72be6fd960,1925.32,8900.03,1867.58,57.73,487,1440,7015.21,1884.82,1
0x27b820e5203aa114acc2712e0e1d0ad758abb68c,27609.69,230217.25,29875.32,-2265.62,561,947,187357.69,42859.57,1
0x2853240a0f4e9e11a949a5cfa6e0fe953a293482,29761.59,230662.67,26247.23,3514.37,1440,3374,175225.20,55437.47,1
0x520b4a0452005517964ee864985f2026d0dc100a,1586.74,8354.05,1586.74,0.00,0,450,8354.05,0.00,1
0x5b7065359a6b6fe34d061b2e5a9bd06f2b5c5e72,1973.45,5739.52,1973.45,0.00,0,59,5739.52,0.00,1
0x672225e5e1aba1c970ac613efd1505f4b7a10762,3742.51,175151.46,3760.60,-18.09,130,1263,173513.30,1638.16,1
0x6b61b495ee20b5a161b7f75649e3b8888b217a33,6048.95,11271.70,5902.74,146.20,8,173,11012.71,258.99,1
0x815faf650a093427c1386f3fc7465d03bdc7f1b3,1897.96,634244.66,1945.13,-47.17,128,655,575857.51,58387.15,1


In [599]:
wallet_cohorts['multi_filter'].sort_values("copyable_pnl", ascending=False)['wallet'].head(20).to_list()

['0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8',
 '0x27b820e5203aa114acc2712e0e1d0ad758abb68c',
 '0x0cb10c40b0776e9ee8cef970af85724654dda76c',
 '0x2853240a0f4e9e11a949a5cfa6e0fe953a293482',
 '0xc757850237e266d3b08d16b8455777a88590ccfc',
 '0x672225e5e1aba1c970ac613efd1505f4b7a10762',
 '0x6b61b495ee20b5a161b7f75649e3b8888b217a33',
 '0x5b7065359a6b6fe34d061b2e5a9bd06f2b5c5e72',
 '0x520b4a0452005517964ee864985f2026d0dc100a',
 '0x20a89c60befe1e2d866042deed93ab72be6fd960',
 '0x99984e22205053950eb25453779267bcc1aee858',
 '0x815faf650a093427c1386f3fc7465d03bdc7f1b3']

In [600]:
df = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)
print(len(df))
WALLETS = set(wallet_cohorts['multi_filter']['wallet'])
df[ (df["dt"] >= pd.Timestamp("2026-03-04 16:00:00", tz="UTC"))
    & (df["dt"] <= pd.Timestamp("2026-03-04 17:00:00", tz="UTC"))
   & df['wallet'].isin(WALLETS)].head(1)

9787589


,wallet,condition_id,dt,side,outcome,position,total_quantity,avg_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count
1190820,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x2d927579f3d6086c028ab69f68a9dc25b62501dca6cc58f783f6aaca8734212f,2026-03-04 16:26:55+00:00,BUY,Yes,3.54,3.54,0.65,2.30,3.54,1.24,0.00,True,1.00,0xa662f0b38db46a000b98ade00c36dff7a66fbb0f9b4da4bba1c3c41de893e18e,1,False,0.00,0.00,0.00


In [601]:
watched_wallets = wallet_cohorts['multi_filter'].sort_values("copyable_pnl", ascending=False)['wallet'].head(20).to_list()
for w in watched_wallets:
    wallet_df = wallet_vol_train[wallet_vol_train['wallet'] == w]
    wallet_cohorts[w] = wallet_df.copy().reset_index(drop=True)

In [627]:
print(len(wallet_cohorts['multi_filter']))
pd.set_option('display.max_rows', 100)
wallet_cohorts["multi_filter"].sort_values("total_pnl", ascending=False).head(10)

12


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top_market_pnl_pct,median_roi,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
4,0x0cb10c40b0776e9ee8cef970af85724654dda76c,1.36,4029.00,1375.00,210082.01,32542.53,5767.43,0.18,0.09,0.01,1.07,1030.77,0.03,695.58,0.12,0.15,0.18,0.19
5,0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8,1.51,2084.00,568.00,163263.85,32073.32,8392.48,0.24,0.07,0.05,0.52,2085.63,0.07,1725.99,0.21,0.20,0.26,0.14
10,0x2853240a0f4e9e11a949a5cfa6e0fe953a293482,0.90,3884.00,1132.00,230662.67,29761.59,4503.21,0.20,0.11,0.02,0.44,744.59,0.03,559.70,0.12,0.13,0.15,0.07
9,0x27b820e5203aa114acc2712e0e1d0ad758abb68c,2.20,789.00,251.00,230217.25,27609.69,7233.23,0.45,0.13,0.01,0.27,1045.61,0.04,1074.73,0.15,0.12,0.26,0.07
11,0x6b61b495ee20b5a161b7f75649e3b8888b217a33,3.52,149.00,25.00,11271.70,6048.95,783.70,0.46,0.27,0.44,0.47,190.05,0.03,33.62,0.04,0.54,0.13,0.06
8,0xc757850237e266d3b08d16b8455777a88590ccfc,1.62,166.00,44.00,1030198.18,5760.79,2787.96,0.32,0.27,0.01,0.17,496.00,0.09,496.00,0.18,0.01,0.48,0.08
1,0x672225e5e1aba1c970ac613efd1505f4b7a10762,1.55,770.00,240.00,175151.46,3742.51,1126.36,0.44,0.16,0.00,1.17,9.00,0.00,6.38,0.01,0.02,0.30,0.35
3,0x99984e22205053950eb25453779267bcc1aee858,0.33,3385.00,1072.00,32667.36,3545.28,482.97,0.23,0.05,-1.00,1.68,126.32,0.04,104.84,0.22,0.11,0.14,0.23
6,0x5b7065359a6b6fe34d061b2e5a9bd06f2b5c5e72,2.02,59.00,57.00,5739.52,1973.45,710.06,0.29,0.06,0.79,0.35,196.00,0.10,108.00,0.15,0.34,0.36,0.13
7,0x20a89c60befe1e2d866042deed93ab72be6fd960,0.51,968.00,334.00,8900.03,1925.32,594.18,0.26,0.07,0.22,0.31,97.65,0.05,89.93,0.15,0.22,0.31,0.10


In [628]:
selected_wallets = wallet_cohorts["multi_filter"]
selected_wallets.to_parquet(WORKSPACE_DIR / "selected_wallets_v2.parquet", index=False)

In [629]:
wallet_cohorts["multi_filter"]['wallet'].to_list()

['0x815faf650a093427c1386f3fc7465d03bdc7f1b3',
 '0x672225e5e1aba1c970ac613efd1505f4b7a10762',
 '0x520b4a0452005517964ee864985f2026d0dc100a',
 '0x99984e22205053950eb25453779267bcc1aee858',
 '0x0cb10c40b0776e9ee8cef970af85724654dda76c',
 '0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8',
 '0x5b7065359a6b6fe34d061b2e5a9bd06f2b5c5e72',
 '0x20a89c60befe1e2d866042deed93ab72be6fd960',
 '0xc757850237e266d3b08d16b8455777a88590ccfc',
 '0x27b820e5203aa114acc2712e0e1d0ad758abb68c',
 '0x2853240a0f4e9e11a949a5cfa6e0fe953a293482',
 '0x6b61b495ee20b5a161b7f75649e3b8888b217a33']

## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [630]:
# from polymarket_analysis.wallet_selection.selector import build_strategies_from_sweep
# from polymarket_analysis.strategy.registry import save_strategy

# all_strategies = build_strategies_from_sweep(
#     wallet_cohorts=wallet_cohorts,
#     signal_threshold=SIGNAL_THRESHOLD,
#     selection_metric=BEST_SELECTION_METRIC,
#     top_n=BEST_TOP_N,
#     sweep_df=cohort_sweep,
#     extra_metadata={
#         "end_date_train": str(END_DATE_TRAIN),
#         "train_a_end_date": str(TRAIN_A_END_DATE),
#     },
# )

# for strategy in all_strategies:
#     parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
#     print(f"Saved [{strategy.strategy_id}]  wallets={len(strategy.wallets):3d}  trigger={strategy.trigger.fn_ref.split('.')[-1]}")

# print(f"\nTotal strategies saved: {len(all_strategies)}")


## Strategy registry summary

In [631]:
# from polymarket_analysis.strategy.registry import load_all_strategies

# registry = load_all_strategies(WORKSPACE_DIR)
# summary_rows = []
# for sid, s in registry.items():
#     summary_rows.append({
#         "strategy_id": s.strategy_id,
#         "name": s.name,
#         "selection_method": s.selection_method,
#         "num_wallets": len(s.wallets),
#         "trigger_fn": s.trigger.fn_ref.split(".")[-1],
#         "threshold": s.trigger.params.get("threshold"),
#         "dynamic_sizing": s.trigger.params.get("dynamic_sizing"),
#     })

# pd.DataFrame(summary_rows)


## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [632]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [633]:
_trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_fills = pd.concat([pd.read_parquet(f) for f in _trade_files], ignore_index=True)
df_fills["dt"]               = pd.to_datetime(df_fills["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

In [634]:
multi_filter_fills = df_fills[
    (df_fills["wallet"].isin(wallet_cohorts["multi_filter"]["wallet"]))
    & (df_fills["is_train"] == False)
].sort_values("dt")
len(multi_filter_fills)

9116

In [635]:
# last_ten = ['0x6fbbd19e716f7454bdf483c41e6fa276a9e139da',
#  '0x47c6e09427e5581445323964afa8432803e82fe4',
#  '0x3f2c9b3d2d7a09143f810940b4da51ac90b2f0cf',
#  '0x8b0244a0ded9943470a6bdad9050dce24bb1f3d9',
#  '0x0c4f5b1295f39cf505679209a22adbafe61c0f33',
#  '0xaf3c7d23f497ea9aa23c522a25fa291274b4f5ff',
#  '0xbff9efe3a976c115e3e639b4c6b9c7168479009d',
#  '0x236599e3745dbea9d285d7ac967846f476d8aaf4',
#  '0xd6f44883f664d7dc963d8b89c5a0689fdd330566',
#  '0xf201a19b43471261a3c1ba9247335d55270e527e']
# top_ten =['0xabde435151e5cf858071cec380303acb610fd41f',
#  '0xec593b3c1a9a0ed15e597dc4ea09cafbffe3bb35',
#  '0x37bdf9681cecfebe1f5709d2aed823e0063c2a26',
#  '0x702226b14b65b1c8d05ff51759e095f09932f64e',
#  '0xcf0aca0d7a395202aec661c3666be9cc098e320a',
#  '0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62',
#  '0x4a4b5d59f1979fdf45fa15c17e2ba414f2ebbbca',
#  '0x2687cb380661a0041bf4c4cf3945a034b79e1363',
#  '0xc5b5bbd42624a8f0c8dfa90221913007d8c77e80',
#  '0x3335de45972fd32e5d8a3b689e772147f2693656']
# vol = wallet_cohorts['volatility']
# last_ten_vol = vol[vol['wallet'].isin(last_ten)].copy().reset_index(drop=True)
# top_ten_vol = vol[vol['wallet'].isin(top_ten)].copy().reset_index(drop=True)
# wallet_cohorts['last_ten'] = last_ten_vol
# wallet_cohorts['top_ten'] = top_ten_vol

In [636]:
_trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_fills = pd.concat([pd.read_parquet(f) for f in _trade_files], ignore_index=True)
df_fills["dt"]               = pd.to_datetime(df_fills["dt"], utc=True)
df_fills['copyable_pnl_exposure'] = np.where( 
    df_fills['copyable_qty'] > 0,
    df_fills['avg_price'] * df_fills['copyable_qty'],
    np.nan
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

len(df_fills), len(df_fills[df_fills["dt"] < split_dt]), len(df_fills[df_fills["dt"] >= split_dt])

(9787589, 7111315, 2676274)

In [637]:
filter_fills = df_fills[
    (df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))
    & (df_fills['side'] == 'BUY')
    # & (df_fills["dt"] >= split_dt)
                        ].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills = filter_fills[filter_fills['copyable_qty'] >= 1].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills.head()

23573
5932


,wallet,condition_id,dt,side,outcome,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count,copyable_pnl_exposure
0,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x0956f8964da8a5e45733c6004a43d33e154a4f2893c9ca81e5b3015e374200f9,2025-01-17 02:44:57+00:00,BUY,Yes,153.00,153.00,0.87,133.11,153.00,19.89,19.89,True,1.00,0x564c12b422805d123dcfb1c2e751921406273355e1fa0ab153989b2d2cc6fc4b,1,True,153.00,1544.66,15.00,133.11
1,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x0d8b2f0df2ba8dafa07b7e9cad17905abe413ed62805b21be899176aa0999aa3,2025-04-11 04:07:45+00:00,BUY,Yes,42.15,16.15,0.26,4.20,0.00,-4.20,-0.33,False,0.00,0xc40b68603c95dc443254d6a0183f1e6d75a8e805fa0ee1f9dfd7d703dc4795da,1,True,1.27,0.27,1.00,0.33
2,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x0a37939eff6e70da165aeae19f2f9d8aaeb3baa73f84f1436ceb48fe21797e98,2025-04-16 13:49:31+00:00,BUY,Yes,50.00,50.00,0.06,3.00,50.00,47.00,4.95,True,1.00,0x4a8ec952241c8a7929e5be75870b5ca8aa4d000b409491d07397b9f9e9357d3f,1,True,5.26,0.26,1.00,0.32
3,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x0beaaf25d4df0a4fbbdda991db8a67a00315a417024f66850a7b94465b125a72,2025-06-05 18:47:18+00:00,BUY,No,200.00,200.00,0.00,0.40,0.00,-0.40,-0.40,False,0.00,0x33021942c3fedd65a7bfab1bbe114d4b24bfa7b5cad85ac2adb261ec439fa8a9,1,True,200.00,1351.60,3.00,0.40
4,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x0c014d0f1d560c6f3fc3f28b2699c4c5894e64fa8e21c5322d599ad1738c2721,2025-07-03 22:07:35+00:00,BUY,Yes,109.89,29.89,0.44,13.15,29.89,16.74,16.74,True,1.00,0xea766ca088eb8c3df0817c46f8e0376588cb8c3ad425ce3b72c2464ddb4c3937,1,True,29.89,25.13,2.00,13.15


In [638]:
pd.set_option('display.max_colwidth', 100)

In [661]:
filter_fills["bucket"] = filter_fills["dt"].dt.floor('5min')

MAX_EXPOSURE = 1

df = filter_fills.groupby(['bucket', 'condition_id', 'wallet']).agg(
    copyable_pnl_exposure=('copyable_pnl_exposure', 'sum'),
    total_copyable_qty=('copyable_qty', 'sum'),
    trade_pnl =('trade_pnl', 'sum'),
    copyable_pnl = ('copyable_pnl', 'sum'),
    trades=('condition_id', 'count'),
    copyable_qty=('copyable_qty', 'sum'),
    wallets=('wallet', lambda x: x.nunique()),
).reset_index()

scale = np.minimum(1, MAX_EXPOSURE / df["copyable_pnl_exposure"].replace(0, np.nan))

df = df.assign(
    scaled_copyable_pnl_exposure = df["copyable_pnl_exposure"] * scale,
    scaled_copyable_pnl = df["copyable_pnl"] * scale,
    scaled_copyable_qty = df["copyable_qty"] * scale,
)

df.head()

,bucket,condition_id,wallet,copyable_pnl_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,copyable_qty,wallets,scaled_copyable_pnl_exposure,scaled_copyable_pnl,scaled_copyable_qty
0,2025-01-02 14:10:00+00:00,0xb64d0eb9636ace08b2e15514a729c8e9ed7468148300912324ecb8ce8416f9db,0x815faf650a093427c1386f3fc7465d03bdc7f1b3,3.00,100.00,-3.00,-3.00,1,100.00,1,1.00,-1.00,33.33
1,2025-01-02 15:45:00+00:00,0xb64d0eb9636ace08b2e15514a729c8e9ed7468148300912324ecb8ce8416f9db,0x815faf650a093427c1386f3fc7465d03bdc7f1b3,1315.24,1346.20,115.09,30.96,1,1346.20,1,1.00,0.02,1.02
2,2025-01-02 16:05:00+00:00,0xb64d0eb9636ace08b2e15514a729c8e9ed7468148300912324ecb8ce8416f9db,0x815faf650a093427c1386f3fc7465d03bdc7f1b3,2.90,100.00,-2.90,-2.90,1,100.00,1,1.00,-1.00,34.48
3,2025-01-05 02:10:00+00:00,0x0048c315a8cb7d0498a7e6292a9b3545368f2d5a40600c20447eff757bb5c16c,0x815faf650a093427c1386f3fc7465d03bdc7f1b3,335.67,336.01,0.34,0.34,2,336.01,1,1.00,0.00,1.00
4,2025-01-06 14:00:00+00:00,0xae692198af3dfba8d161c24a00d1384f927073c7ecc20996cd4892413bd70bb5,0x0cb10c40b0776e9ee8cef970af85724654dda76c,23.00,100.00,-23.00,-23.00,1,100.00,1,1.00,-1.00,4.35


In [662]:
from polymarket_analysis.data.data_catalogue import load_markets
import datetime as dt

# start_date = datetime.datetime(2025, 1, 1)
start_date = split_dt.to_pydatetime()

market_df = load_markets(start_date, dt.datetime.now(dt.timezone.utc))
market_df = market_df[market_df['condition_id'].isin(df['condition_id'])].copy().reset_index(drop=True)

Found 3 market file(s)


In [654]:
len(market_df)

995

In [655]:
market_df.groupby('end_date_iso').size().reset_index(name='count').sort_values('end_date_iso').tail(10)

,end_date_iso,count
46,2026-03-19T00:00:00Z,17
47,2026-03-20T00:00:00Z,18
48,2026-03-21T00:00:00Z,7
49,2026-03-22T00:00:00Z,24
50,2026-03-23T00:00:00Z,14
51,2026-03-24T00:00:00Z,6
52,2026-03-25T00:00:00Z,13
53,2026-03-26T00:00:00Z,20
54,2026-03-27T00:00:00Z,36
55,2026-03-28T00:00:00Z,14


In [663]:
df = (df.merge(market_df, on='condition_id', how='inner')
    .sort_values(['bucket', 'copyable_pnl_exposure'], ascending=[True, False])
    .reset_index(drop=True)
)

In [664]:
df_plot = df.groupby('end_date_iso').agg(
    copyable_pnl=('scaled_copyable_pnl', 'sum'),
    ending_exposure=('scaled_copyable_pnl_exposure', 'sum'),
).reset_index()

df_plot['copyable_pnl_cumsum'] = df_plot['copyable_pnl'].cumsum()

# exposure ends at EOD of end_date, so shift by 24h
df_exposure = df_plot[['end_date_iso', 'ending_exposure']].copy()
df_exposure['end_date_iso'] = pd.to_datetime(df_exposure['end_date_iso']) + pd.Timedelta(days=1)
df_exposure.rename(columns={'end_date_iso': 'dt', 'ending_exposure': 'exposure_delta'}, inplace=True)

resolution_exposure = df_exposure.groupby('dt').agg(exposure_delta=('exposure_delta', 'sum')).reset_index()
resolution_exposure['exposure_delta'] = resolution_exposure['exposure_delta'] * -1

buy_exposure = df[['bucket', 'scaled_copyable_pnl_exposure']].rename(columns={'bucket': 'dt', 'scaled_copyable_pnl_exposure': 'exposure_delta'})

df_exposure = ( pd.concat([resolution_exposure, buy_exposure], ignore_index=True)
    .reset_index(drop=True)
)

df_exposure.sort_values('dt', inplace=True)
df_exposure['exposure'] = df_exposure['exposure_delta'].cumsum()

# df_plot['end_date_iso'] = pd.to_datetime(df_plot['end_date_iso']) + pd.Timedelta(days=1)

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_exposure['dt'],
    y=df_exposure['exposure'],
    mode='lines',
    name='exposure'
))

fig.add_trace(go.Scatter(
    x=df_plot['end_date_iso'],
    y=df_plot['copyable_pnl_cumsum'],
    mode='lines',
    name='copyable_pnl_cumsum'
))

fig.show()

In [665]:
import plotly.express as px

# Aggregate by bucket
bucket_pnl = df.groupby('bucket')['scaled_copyable_pnl'].sum().reset_index()
bucket_pnl['scaled_copyable_pnl'] = bucket_pnl['scaled_copyable_pnl'].cumsum()

# Plot with Plotly
fig = px.line(
    bucket_pnl,
    x='bucket',
    y='scaled_copyable_pnl',
    title='Total Scaled Copyable PnL by Time Bucket',
    labels={'bucket': 'Time Bucket', 'scaled_copyable_pnl': 'Scaled Copyable PnL'}
)
fig.show()

In [666]:
( 
    df.groupby('condition_id').agg(
        total_copyable_exposure=('copyable_pnl_exposure', 'sum'),
        total_copyable_qty=('copyable_qty', 'sum'),
        trade_pnl =('trade_pnl', 'sum'),
        copyable_pnl = ('copyable_pnl', 'sum'),
        trades=('condition_id', 'count'),
        wallets=('wallet', lambda x: x.nunique()),
 ).sort_values('copyable_pnl', ascending=False).quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])
)

,total_copyable_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,wallets
0.01,0.18,1.25,-194.25,-168.30,1.00,1.00
0.10,1.39,4.97,-23.66,-13.56,1.00,1.00
0.20,3.72,8.00,-6.41,-4.79,1.00,1.00
0.30,6.06,13.70,-1.31,-0.88,1.00,1.00
0.40,10.48,21.00,1.00,0.30,1.00,1.00
0.50,17.66,35.74,2.67,1.20,1.00,1.00
0.60,27.05,58.10,5.41,2.77,1.00,1.00
0.70,48.59,93.93,10.00,5.70,2.00,1.00
0.80,98.43,166.00,19.04,11.74,2.00,1.00
0.90,253.55,342.23,50.28,29.94,3.00,1.00


In [671]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period)",
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period)",
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")

else:
    print("PLOT_WALLET_PNL=False — skipping plots")
